# Chapter 9 Computational Lab
## Continuous Distributions and Geometric Probability

This notebook accompanies Chapter 9 of *Probability Theory with Python and AI*.

Chapter 8 studied probability laws concentrated on countable sets. We now study laws described by densities and then use continuous uniform models to solve geometric-probability problems.

### Learning goals

By the end of the lab you should be able to:

1. state the density representation of a cdf;
2. verify whether a non-negative function is normalized to one;
3. compute interval probabilities by integrating a density;
4. explain why endpoints do not matter when a density exists;
5. distinguish probability zero from logical impossibility;
6. explain why a density may exceed one;
7. recover a density from a cdf only where the density is continuous;
8. work with the symmetric triangular density;
9. compute moments from a density using Chapter 7's expectation formula;
10. recognize and simulate the continuous uniform law $U(a,b)$;
11. use affine transformations of a standard uniform variable;
12. formulate geometric probability as normalized area;
13. compute probabilities as areas below continuous graphs;
14. derive Buffon's needle crossing formula for $0<\ell\le d$;
15. distinguish an exact geometric proof from a Monte Carlo simulation;
16. estimate $\pi$ from Buffon data and understand sampling variability;
17. audit AI-generated claims about densities and geometric probability.

> **Terminology rule.** “Has a density” is more precise than “continuous random variable.” A continuous cdf need not be generated by a Riemann density.


## 0. Setup

The chapter uses ordinary Riemann integration on compact intervals and improper Riemann integrals where required. Numerical integration below is only a computational check of the exact formulas.


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def numerical_integral(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)


def triangular_density(x):
    x = np.asarray(x, dtype=float)
    return np.where(
        np.abs(x) <= 1,
        1 - np.abs(x),
        0.0,
    )


def triangular_cdf(x):
    x = np.asarray(x, dtype=float)
    result = np.empty_like(x)

    mask1 = x < -1
    mask2 = (-1 <= x) & (x <= 0)
    mask3 = (0 < x) & (x <= 1)
    mask4 = x > 1

    result[mask1] = 0.0
    result[mask2] = ((x[mask2] + 1) ** 2) / 2
    result[mask3] = 1 - ((1 - x[mask3]) ** 2) / 2
    result[mask4] = 1.0

    return result


def uniform_density(x, a, b):
    x = np.asarray(x, dtype=float)
    return np.where(
        (a <= x) & (x <= b),
        1 / (b-a),
        0.0,
    )


def uniform_cdf(x, a, b):
    x = np.asarray(x, dtype=float)
    return np.where(
        x < a,
        0.0,
        np.where(
            x <= b,
            (x-a)/(b-a),
            1.0,
        ),
    )


def buffon_probability(ell, d):
    if not (0 < ell <= d):
        raise ValueError("This chapter assumes 0 < ell <= d.")
    return 2*ell/(math.pi*d)


def buffon_simulation(number_of_throws, ell=1.0, d=1.0, seed=20260815):
    if not (0 < ell <= d):
        raise ValueError("Require 0 < ell <= d.")

    rng = np.random.default_rng(seed)

    theta = rng.uniform(0, math.pi/2, size=number_of_throws)
    x = rng.uniform(0, d/2, size=number_of_throws)

    crossing = x <= (ell/2)*np.sin(theta)

    return {
        "crossings": int(np.sum(crossing)),
        "frequency": float(np.mean(crossing)),
        "theta": theta,
        "x": x,
        "crossing": crossing,
    }


def pi_hat_from_buffon(ell, d, N, H):
    if H <= 0:
        return float("inf")
    return 2*ell*N/(d*H)


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Continuous-distribution tools are ready."
    "</div>"
))


## 1. Distributions with a density

A non-negative function $f_X$ that is Riemann integrable on every compact interval is a density of $X$ when

$$
F_X(x)
=
\int_{-\infty}^{x}f_X(t)\,dt,
$$

and

$$
\int_{-\infty}^{\infty}f_X(t)\,dt=1.
$$

The density is **not** itself a probability. Probability is obtained by integrating the density over a set.


### Density validity requires two checks

A candidate density must satisfy

$$
f(x)\ge0
$$

and

$$
\int_{-\infty}^{\infty}f(x)\,dx=1.
$$

Non-negativity alone is not enough.


In [ ]:
density_constant = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=2.0,
    step=0.05,
    description="c",
)
density_output = widgets.Output()


def update_density_candidate(*_):
    with density_output:
        clear_output(wait=True)

        c = density_constant.value

        # Candidate: f(x)=c on [0,2], zero elsewhere.
        total_mass = 2*c

        display(Math(
            r"\int_{-\infty}^{\infty}f(x)\,dx="
            + f"{total_mass:.4f}"
        ))

        valid = abs(total_mass - 1) < 1e-12
        display(Markdown(f"**Valid density:** {valid}"))

        if valid:
            display(Math(r"c=\frac12"))


density_constant.observe(update_density_candidate, names="value")
display(widgets.VBox([density_constant, density_output]))
update_density_candidate()


### A density can be larger than one

A probability density is measured in probability per unit of length.

For

$$
X\sim U(0,0.2),
$$

the density is

$$
f_X(x)=5
$$

on $[0,0.2]$.

The value $5$ is not a probability. The total area is

$$
5(0.2)=1.
$$


In [ ]:
x = np.linspace(-0.05, 0.25, 500)
f = np.where((0 <= x) & (x <= 0.2), 5.0, 0.0)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x, f)
ax.fill_between(x, 0, f, alpha=0.2)
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("A legitimate density with height 5")
plt.show()

display(Math(r"\int_0^{0.2}5\,dx=1"))


## 2. Interval probabilities from a density

If $X$ has density $f_X$, then for $a<b$,

$$
\boxed{
P(a<X\le b)
=
\int_a^b f_X(x)\,dx.
}
$$

Because a distribution with a density has no atoms,

$$
P(X=x)=0
$$

for every real $x$.

Therefore all four endpoint conventions agree:

$$
P(a<X\le b)
=
P(a\le X\le b)
=
P(a<X<b)
=
P(a\le X<b).
$$


### Probability zero is not impossibility

For $X\sim U(0,1)$,

$$
P(X=1/2)=0.
$$

This does **not** mean that the numerical value $1/2$ is logically impossible. The singleton simply has zero probability in the continuous model.

The statement

$$
P(A)=0
$$

is a statement about the probability measure, not a declaration that $A=\varnothing$.


In [ ]:
display(Math(r"P(X=1/2)=0"))
display(Markdown(
    "A continuous model can contain non-empty events with probability zero."
))


## 3. Recovering a density from the cdf

If

$$
F_X(x)
=
\int_{-\infty}^{x}f_X(t)\,dt
$$

and $f_X$ is continuous at $x$, then

$$
\boxed{
F_X'(x)=f_X(x).
}
$$

The continuity qualification matters.

Changing a Riemann density at finitely many points does not change any interval integral and therefore does not change the probability law.


### Same law after changing one density value

Define

$$
f(x)=1,
\qquad
0\le x\le1,
$$

and define $g$ to agree with $f$ except that

$$
g(1/2)=100.
$$

The functions differ at one point, but every Riemann integral over an interval is unchanged. They therefore define the same law.


In [ ]:
grid = np.linspace(0, 1, 501)
f = np.ones_like(grid)
g = np.ones_like(grid)

idx = np.argmin(np.abs(grid - 0.5))
g[idx] = 100

display(Markdown(
    "A numerical grid makes the exceptional point visually large, "
    "but the exact Riemann integral is unchanged because one point has zero length."
))


### Continuous cdf does not automatically imply a Riemann density

The existence of a continuous cdf is weaker than the existence of a density.

Singular continuous distributions provide continuous cdfs that are not generated by a Riemann density in the sense used in this chapter.

Therefore the precise language is:

> “the distribution has a density,”

not merely:

> “the random variable is continuous.”


## 4. The triangular density

Consider

$$
f_X(x)
=
\begin{cases}
1-|x|,&|x|\le1,\\
0,&|x|>1.
\end{cases}
$$

Normalization follows from

$$
\int_{-1}^{1}(1-|x|)\,dx
=
2\int_0^1(1-x)\,dx
=
1.
$$


In [ ]:
grid = np.linspace(-1.5, 1.5, 2001)
f = triangular_density(grid)

fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(grid, f)
ax.fill_between(grid, 0, f, alpha=0.2)
ax.set_xlabel("x")
ax.set_ylabel("f_X(x)")
ax.set_title("Triangular density")
plt.show()

mass = numerical_integral(f, grid)
display(Math(r"\text{numerical total mass}\approx" + f"{mass:.8f}"))


### Exact cdf of the triangular density

The cdf is

$$
F_X(x)
=
\begin{cases}
0,&x<-1,\\
\dfrac{(x+1)^2}{2},&-1\le x\le0,\\
1-\dfrac{(1-x)^2}{2},&0<x\le1,\\
1,&x>1.
\end{cases}
$$

For example,

$$
P\left(-\frac12<X<\frac12\right)
=
\frac34.
$$


In [ ]:
grid = np.linspace(-1.5, 1.5, 2001)
F = triangular_cdf(grid)

fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(grid, F)
ax.set_xlabel("x")
ax.set_ylabel("F_X(x)")
ax.set_ylim(-0.03, 1.03)
ax.set_title("CDF of the triangular law")
plt.show()

p = float(triangular_cdf(np.array([0.5]))[0] - triangular_cdf(np.array([-0.5]))[0])
display(Math(
    r"P(-1/2<X<1/2)=" + f"{p:.6f}"
))


### Numerical reconstruction of the cdf

A density can be converted numerically into a cdf by cumulative integration.

This is a computational approximation of

$$
F_X(x)
=
\int_{-\infty}^{x}f_X(t)\,dt.
$$


In [ ]:
grid = np.linspace(-1, 1, 4001)
f = triangular_density(grid)

dx = grid[1] - grid[0]
areas = 0.5 * (f[:-1] + f[1:]) * dx
F_num = np.concatenate([[0.0], np.cumsum(areas)])
F_exact = triangular_cdf(grid)

max_error = np.max(np.abs(F_num - F_exact))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(grid, F_exact, label="exact cdf")
ax.plot(grid, F_num, linestyle="--", label="numerical cumulative integral")
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("F(x)")
ax.set_title("Exact versus numerically reconstructed triangular cdf")
plt.show()

display(Math(
    r"\max_x|F_{\mathrm{num}}(x)-F_{\mathrm{exact}}(x)|\approx"
    + f"{max_error:.8f}"
))


## 5. Expectations and moments from a density

The density formula for expectation was established in Chapter 7.

If $g$ is continuous and

$$
\int_{-\infty}^{\infty}
|g(x)|f_X(x)\,dx<\infty,
$$

then

$$
\boxed{
\mathbb E[g(X)]
=
\int_{-\infty}^{\infty}
g(x)f_X(x)\,dx.
}
$$

This is **not** a new definition of expectation. It is the general expectation from Chapter 7 simplified by the existence of a density.


### Mean and variance from a density

If

$$
\int_{-\infty}^{\infty}x^2f_X(x)\,dx<\infty,
$$

then

$$
\mathbb E[X]
=
\int_{-\infty}^{\infty}xf_X(x)\,dx,
$$

and

$$
\operatorname{Var}(X)
=
\int_{-\infty}^{\infty}x^2f_X(x)\,dx
-
\left(
\int_{-\infty}^{\infty}xf_X(x)\,dx
\right)^2.
$$


### Symmetry and the mean

If

$$
f_X(-x)=f_X(x)
$$

for every $x$, and

$$
\mathbb E|X|<\infty,
$$

then

$$
\boxed{
\mathbb E[X]=0.
}
$$

The integrand $xf_X(x)$ is odd, so its integral over every symmetric interval is zero.


In [ ]:
grid = np.linspace(-1, 1, 20001)
f = triangular_density(grid)

mean = numerical_integral(grid*f, grid)
second = numerical_integral((grid**2)*f, grid)
variance = second - mean**2

display(Math(r"\mathbb E[X]\approx" + f"{mean:.10f}"))
display(Math(r"\mathbb E[X^2]\approx" + f"{second:.10f}"))
display(Math(r"\operatorname{Var}(X)\approx" + f"{variance:.10f}"))
display(Math(r"\text{exact variance}=\frac16"))


## 6. The continuous uniform distribution

For $a<b$,

$$
X\sim U(a,b)
$$

means that $X$ has density

$$
f_X(x)
=
\begin{cases}
\dfrac1{b-a},&a\le x\le b,\\
0,&\text{otherwise}.
\end{cases}
$$

Equal-length subintervals receive equal probabilities.


### Cdf, mean and variance

For $X\sim U(a,b)$,

$$
F_X(x)
=
\begin{cases}
0,&x<a,\\
\dfrac{x-a}{b-a},&a\le x\le b,\\
1,&x>b.
\end{cases}
$$

Also,

$$
\boxed{
\mathbb E[X]=\frac{a+b}{2},
}
$$

and

$$
\boxed{
\operatorname{Var}(X)=\frac{(b-a)^2}{12}.
}
$$


In [ ]:
uniform_a = widgets.FloatSlider(value=-2, min=-10, max=5, step=0.5, description="a")
uniform_b = widgets.FloatSlider(value=3, min=-5, max=10, step=0.5, description="b")
uniform_output = widgets.Output()


def update_uniform(*_):
    with uniform_output:
        clear_output(wait=True)

        a = uniform_a.value
        b = uniform_b.value

        if not a < b:
            display(Markdown("**Require a<b.**"))
            return

        mean = (a+b)/2
        variance = (b-a)**2/12

        display(Math(r"\mathbb E[X]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(X)=" + f"{variance:.6f}"))

        grid = np.linspace(a-1, b+1, 1000)
        f = uniform_density(grid, a, b)
        F = uniform_cdf(grid, a, b)

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot(grid, f)
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.set_title("Uniform density")
        plt.show()

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot(grid, F)
        ax.set_xlabel("x")
        ax.set_ylabel("cdf")
        ax.set_ylim(-0.03, 1.03)
        ax.set_title("Uniform cdf")
        plt.show()


for control in (uniform_a, uniform_b):
    control.observe(update_uniform, names="value")

display(widgets.VBox([
    widgets.HBox([uniform_a, uniform_b]),
    uniform_output,
]))
update_uniform()


### Random point on a line segment

Choose a point uniformly from a line segment of length $10$, and let $X$ be its distance from the left endpoint.

Then

$$
X\sim U(0,10).
$$

Therefore

$$
P(2\le X\le5)
=
\frac{5-2}{10}
=
0.3,
$$

and

$$
\mathbb E[X]=5.
$$


In [ ]:
display(Math(r"P(2\le X\le5)=0.3"))
display(Math(r"\mathbb E[X]=5"))


## 7. Affine construction from a standard uniform variable

If

$$
U\sim U(0,1)
$$

and $a<b$, then

$$
\boxed{
X=a+(b-a)U
}
$$

satisfies

$$
X\sim U(a,b).
$$

This gives a direct simulation method for any interval-uniform distribution.


In [ ]:
affine_a = widgets.FloatSlider(value=-2, min=-10, max=5, step=0.5, description="a")
affine_b = widgets.FloatSlider(value=3, min=-5, max=10, step=0.5, description="b")
affine_N = widgets.IntSlider(value=5000, min=100, max=30000, step=100, description="N")
affine_output = widgets.Output()


def update_affine_uniform(*_):
    with affine_output:
        clear_output(wait=True)

        a = affine_a.value
        b = affine_b.value
        N = affine_N.value

        if not a < b:
            display(Markdown("**Require a<b.**"))
            return

        rng = np.random.default_rng(2026)
        U = rng.random(N)
        X = a + (b-a)*U

        display(Math(
            r"\text{sample mean}\approx" + f"{np.mean(X):.6f}"
        ))
        display(Math(
            r"\text{theoretical mean}=" + f"{(a+b)/2:.6f}"
        ))

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.hist(X, bins=30, density=True)
        ax.axhline(1/(b-a), linestyle="--")
        ax.set_xlabel("x")
        ax.set_ylabel("empirical density")
        ax.set_title("Affine simulation from U(0,1)")
        plt.show()


for control in (affine_a, affine_b, affine_N):
    control.observe(update_affine_uniform, names="value")

display(widgets.VBox([
    widgets.HBox([affine_a, affine_b]),
    affine_N,
    affine_output,
]))
update_affine_uniform()


## 8. Transforming a standard uniform variable

The cdf method from Chapter 6 remains a powerful way to find the distribution of a transformation.

### Example: $Y=\sqrt U$

Let

$$
U\sim U(0,1),
\qquad
Y=\sqrt U.
$$

For $0\le y\le1$,

$$
F_Y(y)
=
P(\sqrt U\le y)
=
P(U\le y^2)
=
y^2.
$$

Therefore, where differentiable,

$$
f_Y(y)=2y,
\qquad
0<y<1.
$$


In [ ]:
grid = np.linspace(0, 1, 1000)
Fy = grid**2
fy = 2*grid

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(grid, Fy)
ax.set_xlabel("y")
ax.set_ylabel("F_Y(y)")
ax.set_title("CDF of Y=sqrt(U)")
plt.show()

display(Math(
    r"\int_0^1 2y\,dy=1"
))


### Example: $Y=2U-1$

If

$$
Y=2U-1,
$$

then

$$
Y\sim U(-1,1).
$$

This is the affine-construction theorem with $a=-1$ and $b=1$.


## 9. A minimal model of geometric probability

Let

$$
R=[a,b]\times[c,d].
$$

For a region $A\subseteq R$ whose ordinary plane area is defined, the uniform geometric model assigns

$$
\boxed{
P(A)
=
\frac{\operatorname{Area}(A)}
{\operatorname{Area}(R)}.
}
$$

The phrase “choose a point at random” does **not** by itself define a probability model.

Uniformity must be stated: probability is proportional to area.


### Area under a graph

Let

$$
R=[a,b]\times[0,H],
$$

and let $h:[a,b]\to[0,H]$ be continuous.

If a point is chosen uniformly from $R$, then

$$
\boxed{
P(0\le Y\le h(X))
=
\frac1{H(b-a)}
\int_a^b h(x)\,dx.
}
$$

This is simply favorable area divided by total rectangle area.


In [ ]:
graph_power = widgets.FloatSlider(value=1.0, min=0.25, max=4.0, step=0.25, description="power")
graph_output = widgets.Output()


def update_area_graph(*_):
    with graph_output:
        clear_output(wait=True)

        power = graph_power.value

        x = np.linspace(0, 1, 1000)
        h = x**power
        area = numerical_integral(h, x)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(x, h)
        ax.fill_between(x, 0, h, alpha=0.2)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title("Favorable region below y=x^a")
        plt.show()

        display(Math(
            r"P(Y\le X^a)=" + f"{area:.6f}"
        ))
        display(Math(
            r"\text{exact}=\frac1{a+1}="
            + f"{1/(power+1):.6f}"
        ))


graph_power.observe(update_area_graph, names="value")
display(widgets.VBox([graph_power, graph_output]))
update_area_graph()


### Unit-square example

If $(X,Y)$ is chosen uniformly from $[0,1]^2$, then the event

$$
Y\le X
$$

is the triangle below the diagonal.

Therefore

$$
P(Y\le X)
=
\int_0^1x\,dx
=
\frac12.
$$


### Uniform point in a rectangle

Choose $(X,Y)$ uniformly from

$$
[0,3]\times[0,2].
$$

The event

$$
Y\le\frac{2X}{3}
$$

is exactly the triangle under the diagonal from $(0,0)$ to $(3,2)$.

Its area is $3$, while the rectangle has area $6$. Hence

$$
P\left(Y\le\frac{2X}{3}\right)
=
\frac12.
$$


In [ ]:
x = np.linspace(0, 3, 500)
h = 2*x/3

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, h)
ax.fill_between(x, 0, h, alpha=0.2)
ax.set_xlim(0, 3)
ax.set_ylim(0, 2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Uniform rectangle: favorable triangular region")
plt.show()

display(Math(r"P(Y\le2X/3)=\frac12"))


### Quarter-circle probability and the appearance of $\pi$

Choose $(X,Y)$ uniformly from the unit square.

The event

$$
X^2+Y^2\le1
$$

is the quarter of the unit disk inside the square.

Its area is

$$
\frac{\pi}{4},
$$

so

$$
P(X^2+Y^2\le1)
=
\frac{\pi}{4}.
$$

Here $\pi$ enters through the geometric area of a circle.


In [ ]:
theta = np.linspace(0, math.pi/2, 500)
x_arc = np.cos(theta)
y_arc = np.sin(theta)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(x_arc, y_arc)
ax.fill_between(x_arc[::-1], 0, y_arc[::-1], alpha=0.2)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_title("Quarter disk inside the unit square")
plt.show()

display(Math(r"P(X^2+Y^2\le1)=\frac{\pi}{4}"))


### Nonlinear region: $XY\le1/2$

Choose $(X,Y)$ uniformly from $[0,1]^2$.

For $0\le x\le1/2$, every $y\in[0,1]$ is favorable.

For $1/2<x\le1$,

$$
y\le\frac1{2x}.
$$

Therefore

$$
P(XY\le1/2)
=
\frac12
+
\int_{1/2}^{1}\frac1{2x}\,dx
=
\frac12+\frac12\log2.
$$


In [ ]:
x1 = np.linspace(0, 0.5, 300)
x2 = np.linspace(0.5, 1, 500)

fig, ax = plt.subplots(figsize=(6, 5))
ax.fill_between(x1, 0, 1, alpha=0.2)
ax.fill_between(x2, 0, 1/(2*x2), alpha=0.2)
ax.plot(x2, 1/(2*x2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Region XY <= 1/2")
plt.show()

exact = 0.5 + 0.5*math.log(2)
display(Math(
    r"P(XY\le1/2)=\frac12+\frac12\log2\approx"
    + f"{exact:.6f}"
))


## 10. Historical problem: Buffon's needle

Parallel lines are separated by distance $d>0$. A needle of length $\ell$ is dropped at random, with

$$
0<\ell\le d.
$$

We want the probability that the needle crosses one of the lines.


### The two relevant coordinates

Only two coordinates are needed:

- $\theta\in[0,\pi/2]$: the acute angle between the needle and the parallel lines;
- $x\in[0,d/2]$: the distance from the needle midpoint to the nearest line.

Under the uniform model, the parameter rectangle is

$$
0\le\theta\le\frac{\pi}{2},
\qquad
0\le x\le\frac d2.
$$

Its total area is

$$
\frac{\pi d}{4}.
$$


### Crossing condition

The perpendicular half-span of the needle is

$$
\frac{\ell}{2}\sin\theta.
$$

A crossing occurs exactly when

$$
\boxed{
x
\le
\frac{\ell}{2}\sin\theta.
}
$$

The favorable area is therefore

$$
\int_0^{\pi/2}
\frac{\ell}{2}\sin\theta\,d\theta
=
\frac{\ell}{2}.
$$

Dividing favorable area by total area gives

$$
\boxed{
P(\text{crossing})
=
\frac{2\ell}{\pi d}.
}
$$


In [ ]:
buffon_ell = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="ell")
buffon_d = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description="d")
buffon_geom_output = widgets.Output()


def update_buffon_geometry(*_):
    with buffon_geom_output:
        clear_output(wait=True)

        ell = buffon_ell.value
        d = buffon_d.value

        if not (0 < ell <= d):
            display(Markdown("**This derivation assumes 0<ell<=d.**"))
            return

        theta = np.linspace(0, math.pi/2, 500)
        boundary = (ell/2)*np.sin(theta)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(theta, boundary)
        ax.fill_between(theta, 0, boundary, alpha=0.2)
        ax.set_xlim(0, math.pi/2)
        ax.set_ylim(0, d/2)
        ax.set_xlabel("theta")
        ax.set_ylabel("x")
        ax.set_title("Buffon parameter rectangle and crossing region")
        plt.show()

        total_area = (math.pi/2)*(d/2)
        favorable_area = ell/2
        probability = favorable_area/total_area

        display(Math(
            r"\text{favorable area}=" + f"{favorable_area:.6f}"
        ))
        display(Math(
            r"\text{total area}=" + f"{total_area:.6f}"
        ))
        display(Math(
            r"P(\text{crossing})=" + f"{probability:.6f}"
        ))


for control in (buffon_ell, buffon_d):
    control.observe(update_buffon_geometry, names="value")

display(widgets.VBox([
    widgets.HBox([buffon_ell, buffon_d]),
    buffon_geom_output,
]))
update_buffon_geometry()


### Special case $\ell=d$

When the needle length equals the line spacing,

$$
P(\text{crossing})
=
\frac2\pi
\approx0.63662.
$$


In [ ]:
display(Math(
    r"\frac2\pi\approx" + f"{2/math.pi:.8f}"
))


## 11. Python laboratory: Buffon's needle simulation

Simulation samples uniformly from the same parameter rectangle used in the proof:

$$
\Theta\sim U(0,\pi/2),
$$

$$
X\sim U(0,d/2).
$$

A simulated throw is classified as a crossing when

$$
X\le\frac{\ell}{2}\sin\Theta.
$$

The simulation is a numerical experiment. It does **not** prove Buffon's formula.


In [ ]:
buffon_N = widgets.IntSlider(
    value=10000,
    min=100,
    max=100000,
    step=100,
    description="throws",
)
buffon_sim_ell = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=3.0,
    step=0.1,
    description="ell",
)
buffon_sim_d = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=3.0,
    step=0.1,
    description="d",
)
buffon_seed = widgets.IntSlider(
    value=2026,
    min=0,
    max=5000,
    description="seed",
)
buffon_sim_output = widgets.Output()


def update_buffon_simulation(*_):
    with buffon_sim_output:
        clear_output(wait=True)

        N = buffon_N.value
        ell = buffon_sim_ell.value
        d = buffon_sim_d.value
        seed = buffon_seed.value

        if not (0 < ell <= d):
            display(Markdown("**Require 0<ell<=d.**"))
            return

        result = buffon_simulation(N, ell, d, seed)
        exact = buffon_probability(ell, d)

        display(Math(
            r"\widehat P(\text{crossing})="
            + f"{result['frequency']:.6f}"
        ))
        display(Math(
            r"P(\text{crossing})="
            + f"{exact:.6f}"
        ))

        if result["crossings"] > 0:
            pi_hat = pi_hat_from_buffon(
                ell,
                d,
                N,
                result["crossings"],
            )
            display(Math(
                r"\widehat\pi="
                + f"{pi_hat:.6f}"
            ))


for control in (
    buffon_N,
    buffon_sim_ell,
    buffon_sim_d,
    buffon_seed,
):
    control.observe(update_buffon_simulation, names="value")

display(widgets.VBox([
    widgets.HBox([buffon_N, buffon_seed]),
    widgets.HBox([buffon_sim_ell, buffon_sim_d]),
    buffon_sim_output,
]))
update_buffon_simulation()


### Sampling variability

If the same experiment is repeated many times, the empirical crossing frequency changes from run to run.

That variation is **sampling variation**, not a change in the theoretical crossing probability.


In [ ]:
repeat_runs = widgets.IntSlider(value=100, min=10, max=300, step=10, description="runs")
repeat_N = widgets.IntSlider(value=10000, min=500, max=30000, step=500, description="N/run")
repeat_output = widgets.Output()


def update_repeated_buffon(*_):
    with repeat_output:
        clear_output(wait=True)

        runs = repeat_runs.value
        N = repeat_N.value

        frequencies = []
        pi_estimates = []

        for j in range(runs):
            result = buffon_simulation(
                N,
                ell=1.0,
                d=1.0,
                seed=1000+j,
            )
            frequencies.append(result["frequency"])

            if result["crossings"] > 0:
                pi_estimates.append(
                    pi_hat_from_buffon(
                        1.0,
                        1.0,
                        N,
                        result["crossings"],
                    )
                )

        frequencies = np.array(frequencies)
        pi_estimates = np.array(pi_estimates)

        display(Markdown(
            f"Crossing frequency: min={frequencies.min():.5f}, "
            f"max={frequencies.max():.5f}, "
            f"mean={frequencies.mean():.5f}, "
            f"sd={frequencies.std(ddof=1):.5f}"
        ))

        display(Markdown(
            f"Pi estimates: min={pi_estimates.min():.5f}, "
            f"max={pi_estimates.max():.5f}, "
            f"mean={pi_estimates.mean():.5f}, "
            f"sd={pi_estimates.std(ddof=1):.5f}"
        ))

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.hist(pi_estimates, bins=20)
        ax.axvline(math.pi, linestyle="--")
        ax.set_xlabel("pi estimate")
        ax.set_ylabel("frequency")
        ax.set_title("Sampling variability of Buffon pi estimates")
        plt.show()


for control in (repeat_runs, repeat_N):
    control.observe(update_repeated_buffon, names="value")

display(widgets.VBox([
    widgets.HBox([repeat_runs, repeat_N]),
    repeat_output,
]))
update_repeated_buffon()


## 12. Estimating $\pi$ from observed Buffon data

If $H$ crossings occur in $N$ throws, solve

$$
\frac{H}{N}
\approx
\frac{2\ell}{\pi d}
$$

for $\pi$:

$$
\boxed{
\widehat\pi
=
\frac{2\ell N}{dH}.
}
$$

For $\ell=3$, $d=6$, $N=10{,}000$ and $H=3{,}210$,

$$
\widehat\pi
=
\frac{10{,}000}{3{,}210}
\approx3.1153.
$$


In [ ]:
data_ell = widgets.FloatSlider(value=3, min=0.1, max=10, step=0.1, description="ell")
data_d = widgets.FloatSlider(value=6, min=0.1, max=10, step=0.1, description="d")
data_N = widgets.IntSlider(value=10000, min=100, max=100000, step=100, description="N")
data_H = widgets.IntSlider(value=3210, min=1, max=100000, step=1, description="H")
data_output = widgets.Output()


def update_pi_estimate(*_):
    with data_output:
        clear_output(wait=True)

        ell = data_ell.value
        d = data_d.value
        N = data_N.value
        H = data_H.value

        if ell > d:
            display(Markdown("**The chapter formula assumes ell<=d.**"))
            return

        if H > N:
            display(Markdown("**Crossings cannot exceed throws.**"))
            return

        estimate = pi_hat_from_buffon(ell, d, N, H)
        exact_cross = buffon_probability(ell, d)
        empirical = H/N

        display(Math(
            r"\widehat P(\text{crossing})="
            + f"{empirical:.6f}"
        ))
        display(Math(
            r"P(\text{crossing})="
            + f"{exact_cross:.6f}"
        ))
        display(Math(
            r"\widehat\pi="
            + f"{estimate:.6f}"
        ))


for control in (data_ell, data_d, data_N, data_H):
    control.observe(update_pi_estimate, names="value")

display(widgets.VBox([
    widgets.HBox([data_ell, data_d]),
    widgets.HBox([data_N, data_H]),
    data_output,
]))
update_pi_estimate()


## 13. Direct area check: $d=2$, $\ell=1$

The parameter rectangle is

$$
0\le\theta\le\frac{\pi}{2},
\qquad
0\le x\le1.
$$

Its area is

$$
\frac{\pi}{2}.
$$

Crossing occurs when

$$
0\le x\le\frac12\sin\theta.
$$

The favorable area is

$$
\int_0^{\pi/2}\frac12\sin\theta\,d\theta
=
\frac12.
$$

Therefore

$$
P(\text{crossing})
=
\frac{1/2}{\pi/2}
=
\frac1\pi.
$$


In [ ]:
display(Math(
    r"P(\text{crossing})=\frac1\pi\approx"
    + f"{1/math.pi:.8f}"
))


## 14. Scaling the needle length

For fixed spacing $d$, Buffon's formula is

$$
P(\text{crossing})
=
\frac{2\ell}{\pi d}.
$$

Therefore, as long as both needle lengths remain at most $d$, doubling $\ell$ doubles the crossing probability.

The derivation in this chapter does not cover $\ell>d$ because a longer needle can cross more than one line and the geometry changes.


In [ ]:
scale_d = 4.0
ell1 = 1.0
ell2 = 2.0

p1 = buffon_probability(ell1, scale_d)
p2 = buffon_probability(ell2, scale_d)

display(Math(r"P_{\ell=1}=" + f"{p1:.6f}"))
display(Math(r"P_{\ell=2}=" + f"{p2:.6f}"))
display(Markdown(f"Doubling verified: **{abs(p2-2*p1) < 1e-12}**"))


## 15. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Density normalization", "density"),
        ("Interval probability", "interval"),
        ("Probability zero", "zero"),
        ("Triangular density", "triangular"),
        ("Uniform law", "uniform"),
        ("Geometric probability", "geometry"),
        ("Buffon", "buffon"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "density",
            "interval",
            "zero",
            "triangular",
            "uniform",
            "geometry",
            "buffon",
        ])

    if kind == "density":
        target = "0.25"
        prompt = "A density is constant c on [0,4] and zero elsewhere. Find c."
        hint = "Normalization requires 4c=1."
        solution = r"c=\frac14."

    elif kind == "interval":
        target = "0.4"
        prompt = "X~U(0,5). Find P(1<=X<=3)."
        hint = "Probability is interval length divided by total length."
        solution = r"P(1\le X\le3)=\frac25=0.4."

    elif kind == "zero":
        target = "no"
        prompt = "If X has a density and P(X=1/2)=0, is X=1/2 logically impossible? yes/no"
        hint = "Probability zero does not imply the event is empty."
        solution = r"\text{No.}"

    elif kind == "triangular":
        target = "0.75"
        prompt = "For f(x)=1-|x| on [-1,1], find P(-1/2<X<1/2)."
        hint = "Integrate symmetrically or use the cdf."
        solution = r"P(-1/2<X<1/2)=\frac34."

    elif kind == "uniform":
        target = "2"
        prompt = "If X~U(-2,6), find E[X]."
        hint = "Use (a+b)/2."
        solution = r"\mathbb E[X]=2."

    elif kind == "geometry":
        target = "0.5"
        prompt = "A point is uniform in the unit square. Find P(Y<=X)."
        hint = "Use the triangular area below the diagonal."
        solution = r"P(Y\le X)=\frac12."

    else:
        target = str(1/math.pi)
        prompt = "Buffon: d=2 and ell=1. Find the crossing probability as a decimal."
        hint = "Use 2 ell /(pi d)."
        solution = r"P(\text{crossing})=\frac1\pi."

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = state["target"].replace(" ", "")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess) - float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Identify the probability model before integrating.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    prompt_output,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_output,
]))

make_exercise()


## 16. AI Audit: densities and geometric probability

Use this checklist on an AI-generated solution.

1. Is a candidate density non-negative?
2. Does it integrate to one?
3. Is a density value being confused with a probability?
4. Is the AI incorrectly requiring $f(x)\le1$?
5. Is an interval probability computed by area under the density?
6. If a density exists, are endpoint choices correctly recognized as irrelevant?
7. Is $P(X=x)=0$ being confused with impossibility?
8. Is a continuous cdf being incorrectly assumed to have a Riemann density?
9. Is $F'(x)=f(x)$ asserted at a point where the chosen density is discontinuous?
10. Is a change to a density at one isolated point being incorrectly treated as a new law?
11. Is the density expectation formula presented as a new definition rather than a consequence of Chapter 7?
12. Is symmetry used only when absolute integrability is sufficient for the mean to exist?
13. In a uniform law, is probability proportional to interval length?
14. In geometric probability, has “chosen uniformly” been explicitly specified?
15. Is favorable area divided by the correct total area?
16. In Buffon's problem, are the correct coordinates $x$ and $\theta$ used?
17. Is the crossing inequality $x\le(\ell/2)\sin\theta$ correct?
18. Is the chapter restriction $0<\ell\le d$ respected?
19. Is simulation being presented as evidence rather than proof?
20. Is the discrepancy of an empirical $\widehat\pi$ from $\pi$ correctly attributed to sampling variation?

### Claims to audit

- “A density can never exceed $1$.”
- “If $P(X=x)=0$, then $x$ cannot occur.”
- “Every continuous cdf has a Riemann density.”
- “A Buffon simulation proves the exact formula $2\ell/(\pi d)$.”

All four claims are false.


### Suggested AI-guided activities

- “Give me a piecewise non-negative function with one unknown normalizing constant. Make me normalize it, construct its cdf, compute an interval probability and then compute the mean.”
- “Challenge me to explain why a density value may exceed one without violating the probability axioms.”
- “Ask me to distinguish a continuous cdf from a distribution that has a density.”
- “Guide me through Buffon's needle from the parameter rectangle to the crossing inequality and favorable area before revealing the final formula.”
- “Run repeated Buffon simulations and require me to separate theoretical probability, empirical frequency and sampling variability.”


## 17. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. A probability density must:",
        [
            "Choose...",
            "be non-negative and integrate to one",
            "be at most one everywhere",
            "be strictly positive everywhere",
        ],
        "be non-negative and integrate to one",
        r"f\ge0\text{ and }\int f=1.",
    ),
    (
        "2. A density may exceed one:",
        ["Choose...", "true", "false"],
        "true",
        r"\text{Density values are not probabilities.}",
    ),
    (
        "3. If X has a density, then P(X=x):",
        ["Choose...", "equals zero", "must be positive", "equals f(x)"],
        "equals zero",
        r"P(X=x)=0.",
    ),
    (
        "4. Probability zero means logical impossibility:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{A non-empty singleton may have probability zero.}",
    ),
    (
        "5. F'(x)=f(x) is guaranteed when:",
        [
            "Choose...",
            "f is continuous at x",
            "F is merely monotone",
            "f(x)<=1",
        ],
        "f is continuous at x",
        r"\text{Use the fundamental theorem at continuity points of the density.}",
    ),
    (
        "6. Changing a Riemann density at one point:",
        [
            "Choose...",
            "changes the law",
            "does not change the law",
        ],
        "does not change the law",
        r"\text{One point has zero Riemann-integral contribution.}",
    ),
    (
        "7. For X~U(a,b), E[X] equals:",
        [
            "Choose...",
            "(a+b)/2",
            "b-a",
            "1/(b-a)",
        ],
        "(a+b)/2",
        r"\mathbb E[X]=(a+b)/2.",
    ),
    (
        "8. Uniform geometric probability in a rectangle is:",
        [
            "Choose...",
            "favorable area / total area",
            "favorable perimeter / total perimeter",
        ],
        "favorable area / total area",
        r"P(A)=\operatorname{Area}(A)/\operatorname{Area}(R).",
    ),
    (
        "9. Buffon's short-needle crossing condition is:",
        [
            "Choose...",
            "x <= (ell/2) sin(theta)",
            "x <= ell cos(theta)",
            "theta <= x",
        ],
        "x <= (ell/2) sin(theta)",
        r"x\le\frac{\ell}{2}\sin\theta.",
    ),
    (
        "10. For 0<ell<=d, Buffon's crossing probability is:",
        [
            "Choose...",
            "2 ell /(pi d)",
            "ell/d",
            "pi ell/d",
        ],
        "2 ell /(pi d)",
        r"P(\text{crossing})=\frac{2\ell}{\pi d}.",
    ),
    (
        "11. A Buffon simulation is a proof of the exact formula:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Simulation is numerical evidence, not a proof.}",
    ),
    (
        "12. A continuous cdf must have a Riemann density:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Singular continuous laws show otherwise.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="470px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:650px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 18. Automatic mathematical verification

The final code cell checks the central numerical identities from the chapter.


In [ ]:
# Triangular density normalization.
grid = np.linspace(-1, 1, 50001)
f = triangular_density(grid)
assert abs(numerical_integral(f, grid) - 1) < 1e-8

# Triangular interval probability.
p_mid = (
    float(triangular_cdf(np.array([0.5]))[0])
    - float(triangular_cdf(np.array([-0.5]))[0])
)
assert abs(p_mid - 0.75) < 1e-12

# Triangular mean and variance.
mean = numerical_integral(grid*f, grid)
second = numerical_integral((grid**2)*f, grid)
variance = second - mean**2

assert abs(mean) < 1e-10
assert abs(variance - 1/6) < 1e-8

# Uniform formulas.
a, b = -2, 6
assert (a+b)/2 == 2
assert abs((b-a)**2/12 - 16/3) < 1e-12
assert abs((3-1)/(5-0) - 0.4) < 1e-12

# Affine construction range.
rng = np.random.default_rng(123)
U = rng.random(10000)
X = a + (b-a)*U
assert np.all(X >= a)
assert np.all(X <= b)

# Geometric probability.
assert abs(0.5 - numerical_integral(np.linspace(0,1,5001), np.linspace(0,1,5001))) < 1e-5

# Nonlinear unit-square region.
xy_prob = 0.5 + 0.5*math.log(2)
assert 0.5 < xy_prob < 1

# Buffon formula.
assert abs(buffon_probability(1,2) - 1/math.pi) < 1e-12
assert abs(buffon_probability(1,1) - 2/math.pi) < 1e-12

# Buffon favorable-area calculation.
ell, d = 1, 2
theta = np.linspace(0, math.pi/2, 100001)
boundary = (ell/2)*np.sin(theta)
favorable = numerical_integral(boundary, theta)
total = (math.pi/2)*(d/2)

assert abs(favorable - ell/2) < 1e-10
assert abs(favorable/total - 1/math.pi) < 1e-10

# Book-data pi estimate.
pi_estimate = pi_hat_from_buffon(3, 6, 10000, 3210)
assert abs(pi_estimate - 10000/3210) < 1e-12

show_result(
    "All Chapter 9 automatic checks passed",
    r"\int f_X(x)\,dx=1",
    r"P(a<X<b)=\int_a^b f_X(x)\,dx",
    r"\operatorname{Var}(X_{\mathrm{triangular}})=\frac16",
    r"\mathbb E[U(a,b)]=\frac{a+b}{2}",
    r"P(\text{Buffon crossing})=\frac{2\ell}{\pi d}",
    note=(
        "Density normalization, triangular moments, uniform formulas, "
        "geometric-area calculations and Buffon's formula all passed."
    ),
)


## 19. Chapter map

| Chapter concept | Computational representation |
|---|---|
| density | non-negativity plus normalization |
| density is not probability | valid density with height greater than one |
| interval probability | numerical area under $f_X$ |
| zero point masses | probability-zero singleton discussion |
| cdf from density | cumulative integration |
| density from cdf | derivative at continuity points |
| density modification at one point | same-law discussion |
| triangular density | exact density, cdf, moments and reconstruction |
| expectation from density | Chapter 7 formula used computationally |
| symmetry | zero mean for an integrable symmetric law |
| $U(a,b)$ | density, cdf, mean and variance |
| affine uniform construction | simulation from $U(0,1)$ |
| transformations | $Y=\sqrt U$ and $Y=2U-1$ |
| geometric probability | normalized area in a rectangle |
| area below a graph | interactive region calculation |
| unit-square events | triangles, quarter disks and $XY\le1/2$ |
| Buffon's needle | parameter rectangle and crossing inequality |
| Buffon formula | exact favorable-area derivation |
| simulation | empirical crossing frequencies |
| estimating $\pi$ | inversion of the crossing formula |
| sampling variability | repeated simulations |
| AI Audit | density, geometry and proof-vs-simulation checks |

The main conceptual distinction is:

$$
\boxed{
\text{exact probability model and proof}
\quad\ne\quad
\text{numerical simulation of that model}.
}
$$
